# 第 7 週 Part B：診所掛號系統（個人作品）

**規則：** 個人獨立完成；可以使用任何 AI 工具；**禁止與其他同學討論**。

**流程（照順序做）：**
1. 在下一格寫好**規格書**（可以請 AI 幫忙寫初稿，但你要讀過並修改成你要的樣子）。
2. 把規格書交給 AI，請它依照規格產生程式，貼到下面對應的格子。
3. 執行**測試清單**，看懂每一句的 AI 原始回覆與系統解讀。
4. 照最後一格的**交件前檢查清單**交件。

**系統需求（簡化版）：** 科別只有內科、外科、牙科，每科每天有固定名額。功能有掛號、取消、查詢剩餘名額。
🔴 **安全規定：系統不可以提供任何醫療建議或根據症狀推薦科別。** 使用者沒說出科別時，請他直接說要掛哪一科，或洽詢櫃台。

# 📋 規格書（請直接在這一格修改）

學號：411303130　姓名：徐晣紘

## 1. 系統目的（一句話）
讓民眾用自然語言（一句話）完成診所門診的預約掛號、取消掛號與門診名額查詢。

## 2. 使用者會怎麼說（至少 5 句）
1. 我要掛號今天的內科。

2. 張醫師今晚有看診嗎？

3. 幫我取消明天的皮膚科掛號。

4. 請問這週五下午的小兒科還有名額嗎？

5. 我想預約後天早上王醫師的門診。

## 3. 資料表（最多 2 張）
| 表名 | 欄位 | 型態 | 說明 |
|---|---|---|---|
| 門診班表（schedules） | schedule_id | INT | 診次唯一識別碼 (Primary Key) |
| 門診班表（schedules） | doctor_name | VARCHAR | 醫師姓名（如：張醫師、王醫師） |
| 門診班表（schedules） | department | VARCHAR | 科別（內科、小兒科、皮膚科） |
| 門診班表（schedules） | clinic_date | DATE | 看診日期 (YYYY-MM-DD) |
| 門診班表（schedules） | time_slot | VARCHAR | 時段（早上、下午、晚上） |
| 預約紀錄（appointments） | appointment_id | INT | 預約號碼 (Primary Key) |
| 預約紀錄（appointments） | user_phone | VARCHAR | 使用者手機號碼 |
| 預約紀錄（appointments） | schedule_id | INT | 對應的診次識別碼 (Foreign Key) |
| 預約紀錄（appointments） | status | VARCHAR | 狀態（已預約 / 已取消） |
| 預約紀錄（appointments） | created_at | DATETIME | 建檔時間 |
初始資料：

## 4. AI 的工作
- 輸入：使用者的一句話
- 輸出格式：`動作,對象,日期,時段`（例：`掛號,內科,今天,晚上`）
- 動作只能是：掛號、取消、查詢
- 對象只能是：內科、外科、牙科、張醫師、王醫師、李醫師
- 無法判斷時輸出：`未知,未知,未知,未知`
- AI 不可以做的事：
1. 不得直接存取或修改資料庫。

2. 不得替使用者決定或臆測未說出的日期與科別。

3. 不得給出醫療診斷、用藥建議或答覆無關掛號的聊天內容。
## 5. 功能與規則
| 功能 | 條件 | 系統回應 | 資料庫變化 |
|---|---|---|---|
| 門診查詢 | 辨識為查詢且該診次存在 | 「[日期][時段][科別/醫師] 目前剩餘 [X] 個名額。」 | 無變化 |
| 預約掛號 | 辨識為掛號且 available_spots > 0 | 「預約成功！您的預約號碼為 [appointment_id]。」 | available_spots 減 1，appointments 新增 1 筆紀錄 |
| 取消預約 | 辨識為取消且查有已預約紀錄 | 「已成功為您取消 [日期][時段] 的預約。」 | available_spots 加 1，對應紀錄 status 改為 已取消 |

## 6. 例外情況（至少 3 個）
1. 名額已滿：使用者欲掛號的診次 available_spots = 0 時，提示「該診次名額已滿，請選擇其他時段或日期」。
2. 查無預約紀錄：使用者要求取消掛號，但系統找不到對應的手機號碼與預約紀錄時，提示「查無您的預約紀錄，請確認後再試」。
3. 意圖/資訊模糊：使用者輸入「我想看醫生」或「掛號」，未包含明確科別或日期時，提示「請提供您想掛號的科別（如：內科）與日期時段」。

## 7. 測試清單（至少 8 句，正常／錯誤／邊界都要有）
| # | 類型 | 使用者說的話 | 預期結果 |
|---|---|---|---|
| 1 | 正常 | 我要掛號今天的內科 | AI 輸出 掛號,內科,今天,未知；系統完成掛號並扣減名額 |
| 2 | 正常 | 幫我查詢明天下午皮膚科還有沒有名額 |AI 輸出 查詢,皮膚科,明天,下午；系統回覆剩餘名額 12 個 |
| 3 | 正常 | 我想取消張醫師今晚的門診 | AI 輸出 取消,張醫師,今天,晚上；系統更新預約狀態為已取消 |
| 4 | 錯誤 | 請問你們開到幾點？ | AI 輸出 未知,未知,未知,未知；系統提示「無法辨識需求，請說明欲掛號的科別與日期 |
| 5 | 錯誤 | 我肚子很痛該吃什麼藥？ | AI 輸出 未知,未知,未知,未知；系統提示「無法提供醫療建議，若身體不適請儘速就醫」 |
| 6 | 邊界 | 我要掛號明天早上的小兒科（名額已滿） | AI 輸出 掛號,小兒科,明天,早上；系統攔截並回應「該診次名額已滿」 |
| 7 | 邊界 | 我要取消後天的內科預約（無預約紀錄） | AI 輸出 取消,內科,後天,未知；系統回應「查無您的預約紀錄」 |
| 8 | 邊界 | 幫我掛號下禮拜八的骨科 | AI 輸出 未知,未知,未知,未知（或科別不符）；系統提示「目前僅提供內科、小兒科、皮膚科之掛號」 |

## 🤖 可以使用的 prompt

**① 請 AI 寫規格書初稿**
```
我要做一個「診所掛號系統」的課堂小作品（Google Colab + SQLite + Groq + Gradio）。
請依照下面的規格範本，幫我寫出一份規格書初稿。要求：
1. 科別只有：內科、外科、牙科，每科每天有固定名額。
2. 功能：掛號、取消掛號、查詢某科剩餘名額。
3. 資料表不超過 2 張。
4. AI 只負責判斷「動作」與「科別」。如果使用者只描述症狀、沒有說出科別，AI 一律回答未知；系統絕對不可以提供任何醫療建議或推薦科別。
5. 測試清單至少 8 句，要包含正常、錯誤、邊界三種情況，並寫出預期結果。
規格範本如下：
（貼上規格範本）
```

**② 請 AI 依規格寫程式**
```
你是 Python 助教。請依照下面的規格，幫我寫一份 Google Colab 程式，並遵守以下要求：
1. 第 1 格（安裝套件與讀取金鑰）我已經有了，請從第 2 格開始寫，依序為：(2) 建立資料庫與初始資料 (3) 顯示資料表 (4) AI 解讀函式 (5) 測試清單 (6) 功能函式 (7) 主流程與完整流程測試 (8) Gradio 網頁。每格分開給我，開頭用註解說明這格在做什麼。
2. 第 1 格已經定義好 client（Groq）與 MODEL，也已經 import 了 sqlite3、pandas as pd、display、gradio as gr，請直接使用，不要重新讀取金鑰，也不要把金鑰寫在程式裡。
3. 呼叫 Groq 時使用 temperature=0，並用 try/except 包起來，失敗時回傳「未知」，不可讓程式當掉。
4. 每次執行都要先 DROP TABLE IF EXISTS，再重建資料表並放入初始資料。
5. 所有 SQL 都用 ? 參數，不可以用 f-string 組 SQL。
6. 不可直接相信 AI 的回覆：要從回覆中找出允許清單裡的值，找不到就當作「未知」。
7. fetchone() 之後一定要先檢查是否為 None。
8. 資料庫連線用 try/finally 確保關閉。
9. 測試清單的格式為 (使用者說的話, 預期動作, 預期科別)，跑完要印出 AI 原始回覆、系統解讀，以及通過幾句。
10. 只用 sqlite3、groq、gradio、pandas，不要用其他套件。註解用繁體中文，寫得讓初學者看得懂。

規格如下：
（貼上你的規格書）
```

**③ 出錯時請 AI 協助除錯**
```
我的 Colab 程式出現以下錯誤。請先用兩三句話解釋原因，再告訴我「只需要修改哪幾行」，不要整份重寫。
【錯誤訊息】
（貼上完整錯誤訊息）
【出錯的那一格程式】
（貼上程式）
```

In [1]:
# ============================================================
# 第 1 格：安裝套件、讀取金鑰、設定模型
# ============================================================
!pip install -q groq gradio

import sqlite3
import pandas as pd
from IPython.display import display
from google.colab import userdata
from groq import Groq
import gradio as gr

# 🔴 金鑰只能從左側「🔑 密鑰（Secrets）」讀取，絕對不可以寫成 api_key="gsk_..."
#    寫死的金鑰一旦存到 GitHub，就等於公開給全世界使用。
#    也不要 print 出金鑰，因為輸出結果也會一起存到 GitHub。
try:
    client = Groq(api_key=userdata.get('GROQ_API_KEY'))
except Exception as e:
    print("❌ 讀不到金鑰。請檢查：")
    print("   1. 左側 🔑 是否新增了名稱為 GROQ_API_KEY 的密鑰（大小寫要完全一樣）")
    print("   2. 該密鑰的「筆記本存取權」開關是否已打開")
    raise

# 模型名稱集中放在這裡；日後模型更換或下架，只要改這一行
MODEL = "openai/gpt-oss-120b"
print("✅ Groq 金鑰讀取成功，使用模型：", MODEL)

✅ Groq 金鑰讀取成功，使用模型： openai/gpt-oss-120b


In [2]:
# ============================================================
# 第 2 格：建立資料庫與初始資料

import sqlite3

# 建立資料庫連線
conn = sqlite3.connect('clinic.db')
cursor = conn.cursor()

try:
    # 每次執行先刪除舊表格確保資料乾淨
    cursor.execute('DROP TABLE IF EXISTS appointments')
    cursor.execute('DROP TABLE IF EXISTS schedules')

    # 建立門診班表資料表
    cursor.execute('''
        CREATE TABLE schedules (
            schedule_id INTEGER PRIMARY KEY AUTOINCREMENT,
            doctor_name TEXT,
            department TEXT,
            clinic_date TEXT,
            time_slot TEXT,
            max_capacity INTEGER,
            available_spots INTEGER
        )
    ''')

    # 建立預約紀錄資料表
    cursor.execute('''
        CREATE TABLE appointments (
            appointment_id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_phone TEXT,
            schedule_id INTEGER,
            status TEXT,
            created_at DATETIME,
            FOREIGN KEY (schedule_id) REFERENCES schedules (schedule_id)
        )
    ''')

    # 放入初始資料 (schedules)
    initial_schedules = [
        ('張醫師', '內科', '2026-10-29', '晚上', 30, 5),
        ('王醫師', '外科', '2026-10-30', '早上', 30, 0),
        ('李醫師', '牙科', '2026-10-30', '下午', 30, 12)
    ]
    cursor.executemany('''
        INSERT INTO schedules (doctor_name, department, clinic_date, time_slot, max_capacity, available_spots)
        VALUES (?, ?, ?, ?, ?, ?)
    ''', initial_schedules)

    # 放入初始資料 (appointments)
    initial_appointments = [
        ('0912345678', 1, '已預約', '2026-10-28 10:00:00')
    ]
    cursor.executemany('''
        INSERT INTO appointments (user_phone, schedule_id, status, created_at)
        VALUES (?, ?, ?, ?)
    ''', initial_appointments)

    conn.commit()
    print("資料庫建立成功，初始資料已載入！")
finally:
    conn.close()
# 提醒：每次執行都要 DROP TABLE IF EXISTS 再重建
# ============================================================
# （把 AI 產生、你看過並理解的程式貼在這裡）

資料庫建立成功，初始資料已載入！


In [3]:
# ============================================================
# 第 3 格：顯示資料表
import sqlite3
import pandas as pd
from IPython.display import display

# 連線並讀取資料表以供檢查
conn = sqlite3.connect('clinic.db')
try:
    print("【門診班表 schedules】")
    df_schedules = pd.read_sql_query("SELECT * FROM schedules", conn)
    display(df_schedules)

    print("\n【預約紀錄 appointments】")
    df_appointments = pd.read_sql_query("SELECT * FROM appointments", conn)
    display(df_appointments)
finally:
    conn.close()
# 提醒：定義 show_tables() 並呼叫一次
# ============================================================
# （把 AI 產生、你看過並理解的程式貼在這裡）

【門診班表 schedules】


,schedule_id,doctor_name,department,clinic_date,time_slot,max_capacity,available_spots
0,1,張醫師,內科,2026-10-29,晚上,30,5
1,2,王醫師,外科,2026-10-30,早上,30,0
2,3,李醫師,牙科,2026-10-30,下午,30,12



【預約紀錄 appointments】


,appointment_id,user_phone,schedule_id,status,created_at
0,1,0912345678,1,已預約,2026-10-28 10:00:00


In [4]:
# ============================================================
# 第 4 格：AI 解讀函式

def parse_user_input(user_text):
    # 定義允許的清單（防呆驗證用）
    allowed_actions = ["掛號", "取消", "查詢"]
    allowed_objects = ["內科", "外科", "牙科", "張醫師", "王醫師", "李醫師"]
    allowed_dates = ["今天", "明天", "後天", "未知"]
    allowed_slots = ["早上", "下午", "晚上", "未知"]

    prompt = f"""
你是一個診所掛號系統的 AI 助理。請從使用者的話語中提取四個欄位：動作,對象,日期,時段。
- 動作只能是：掛號、取消、查詢 其中之一。若無法判斷填「未知」。
- 對象只能是：內科、外科、牙科、張醫師、王醫師、李醫師 其中之一。若無法判斷填「未知」。
- 日期請判斷為：今天、明天、後天 其中之一。若無法判斷填「未知」。
- 時段請判斷為：早上、下午、晚上 其中之一。若無法判斷填「未知」。
- 絕對不要提供任何醫療建議或推薦科別。若使用者只描述症狀，一律回傳「未知,未知,未知,未知」。
- 輸出格式嚴格規定為：動作,對象,日期,時段（例如：掛號,內科,今天,晚上），不要輸出其他多餘文字。

使用者說的話：{user_text}
"""
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        raw_output = response.choices[0].message.content.strip()
    except Exception as e:
        print(f"AI 呼叫失敗: {e}")
        return "未知,未知,未知,未知", "未知", "未知", "未知", "未知"

    # 解析回覆結果
    parts = raw_output.split(',')
    if len(parts) >= 4:
        action = parts[0].strip()
        obj = parts[1].strip()
        date_val = parts[2].strip()
        slot = parts[3].strip()
    else:
        action, obj, date_val, slot = "未知", "未知", "未知", "未知"

    # 檢查是否在允許清單內（不可直接相信 AI）
    if action not in allowed_actions:
        action = "未知"
    if obj not in allowed_objects:
        obj = "未知"
    if date_val not in allowed_dates:
        date_val = "未知"
    if slot not in allowed_slots:
        slot = "未知"

    return raw_output, action, obj, date_val, slot
# 提醒：回傳 (動作, 科別, 原始回覆)；不可直接相信 AI 的回覆
# ============================================================
# （把 AI 產生、你看過並理解的程式貼在這裡）

In [5]:
# ============================================================
# 第 5 格：測試清單
# 格式為：(使用者說的話, 預期動作, 預期對象/科別)
test_cases = [
    ("我要掛號今天的內科", "掛號", "內科"),
    ("幫我查詢明天下午牙科還有沒有名額", "查詢", "牙科"),
    ("我想取消張醫師今晚的門診", "取消", "張醫師"),
    ("請問你們開到幾點？", "未知", "未知"),
    ("我肚子很痛該吃什麼藥？", "未知", "未知"),
    ("我要掛號明天早上的外科", "掛號", "外科"),
    ("我要取消後天的內科預約", "取消", "內科"),
    ("幫我掛號下禮拜八的骨科", "未知", "未知")
]

print(f"測試清單載入完成，共 {len(test_cases)} 筆測試案例。")
# 提醒：至少 8 句，要與規格書第 7 節一致
# ============================================================
# （把 AI 產生、你看過並理解的程式貼在這裡）

測試清單載入完成，共 8 筆測試案例。


In [6]:
# ============================================================
# 第 6 格：功能函式（掛號、取消、查詢）
# (6) 功能函式
def handle_system_logic(user_text):
    raw_output, action, obj, date_val, slot = parse_user_input(user_text)

    # 若無法判斷動作或對象
    if action == "未知" or obj == "未知":
        if "藥" in user_text or "痛" in user_text or "病" in user_text:
            return raw_output, f"{action},{obj},{date_val},{slot}", "無法提供醫療建議，若身體不適請儘速就醫"
        return raw_output, f"{action},{obj},{date_val},{slot}", "無法辨識需求，請說明欲掛號的科別與日期時段"

    conn = sqlite3.connect('clinic.db')
    cursor = conn.cursor()
    try:
        # 1. 查詢功能
        if action == "查詢":
            cursor.execute("SELECT clinic_date, time_slot, department, doctor_name, available_spots FROM schedules WHERE department = ? OR doctor_name = ?", (obj, obj))
            rows = cursor.fetchall()
            if not rows:
                return raw_output, f"{action},{obj},{date_val},{slot}", f"找不到與「{obj}」相關的門診資訊。"

            result_msgs = []
            for row in rows:
                c_date, t_slot, dept, doc, spots = row
                doc_str = doc if doc else dept
                result_msgs.append(f"{c_date} {t_slot} {doc_str} 目前剩餘 {spots} 個名額。")
            return raw_output, f"{action},{obj},{date_val},{slot}", " ".join(result_msgs)

        # 2. 掛號功能
        elif action == "掛號":
            cursor.execute("SELECT schedule_id, available_spots, clinic_date, time_slot FROM schedules WHERE department = ? OR doctor_name = ?", (obj, obj))
            row = cursor.fetchone()

            # 檢查 fetchone() 是否為 None
            if row is None:
                return raw_output, f"{action},{obj},{date_val},{slot}", f"找不到「{obj}」的門診班表。"

            schedule_id, available_spots, c_date, t_slot = row
            if available_spots <= 0:
                return raw_output, f"{action},{obj},{date_val},{slot}", "該診次名額已滿，請選擇其他時段或日期。"

            # 更新名額並新增預約紀錄（使用 ? 參數）
            cursor.execute("UPDATE schedules SET available_spots = available_spots - 1 WHERE schedule_id = ?", (schedule_id,))
            cursor.execute("INSERT INTO appointments (user_phone, schedule_id, status, created_at) VALUES (?, ?, ?, datetime('now'))",
                           ('0912345678', schedule_id, '已預約'))
            conn.commit()

            cursor.execute("SELECT last_insert_rowid()")
            app_id_row = cursor.fetchone()
            app_id = app_id_row[0] if app_id_row else "未知"

            return raw_output, f"{action},{obj},{date_val},{slot}", f"預約成功！您的預約號碼為 {app_id}。"

        # 3. 取消功能
        elif action == "取消":
            cursor.execute("""
                SELECT a.appointment_id, a.schedule_id FROM appointments a
                JOIN schedules s ON a.schedule_id = s.schedule_id
                WHERE (s.department = ? OR s.doctor_name = ?) AND a.status = '已預約'
            """, (obj, obj))
            row = cursor.fetchone()

            # 檢查 fetchone() 是否為 None
            if row is None:
                return raw_output, f"{action},{obj},{date_val},{slot}", "查無您的預約紀錄，請確認後再試。"

            app_id, schedule_id = row
            cursor.execute("UPDATE appointments SET status = '已取消' WHERE appointment_id = ?", (app_id,))
            cursor.execute("UPDATE schedules SET available_spots = available_spots + 1 WHERE schedule_id = ?", (schedule_id,))
            conn.commit()

            return raw_output, f"{action},{obj},{date_val},{slot}", f"已成功為您取消 {obj} 的預約。"

    finally:
        conn.close()

    return raw_output, f"{action},{obj},{date_val},{slot}", "未知的系統指令。"
# 提醒：fetchone() 後先檢查 None；用 try/finally 關閉連線
# ============================================================
# （把 AI 產生、你看過並理解的程式貼在這裡）

In [7]:
# ============================================================
# 第 7 格：主流程與完整流程測試
passed_count = 0

print("=== 開始執行自動化測試清單 ===")
for idx, (user_said, expected_action, expected_obj) in enumerate(test_cases, 1):
    raw_res, parsed_res, system_reply = handle_system_logic(user_said)

    # 檢查動作與對象是否符合預期
    actions_match = expected_action in parsed_res
    objs_match = expected_obj in parsed_res

    is_passed = actions_match and objs_match
    if is_passed:
        passed_count += 1
        status_str = "【通過 ✅】"
    else:
        status_str = "【失敗 ❌】"

    print(f"\n測試編號 {idx}: {status_str}")
    print(f"  使用者說: {user_said}")
    print(f"  預期: 動作={expected_action}, 對象={expected_obj}")
    print(f"  AI 原始回覆: {raw_res}")
    print(f"  系統解讀: {parsed_res}")
    print(f"  系統回應: {system_reply}")

print(f"\n=== 測試結束 ===")
print(f"總共執行 {len(test_cases)} 句，成功通過 {passed_count} 句。")
# 提醒：最後呼叫 show_tables() 確認資料庫有變化
# ============================================================
# （把 AI 產生、你看過並理解的程式貼在這裡）

=== 開始執行自動化測試清單 ===

測試編號 1: 【通過 ✅】
  使用者說: 我要掛號今天的內科
  預期: 動作=掛號, 對象=內科
  AI 原始回覆: 掛號,內科,今天,未知
  系統解讀: 掛號,內科,今天,未知
  系統回應: 預約成功！您的預約號碼為 2。

測試編號 2: 【通過 ✅】
  使用者說: 幫我查詢明天下午牙科還有沒有名額
  預期: 動作=查詢, 對象=牙科
  AI 原始回覆: 查詢,牙科,明天,下午
  系統解讀: 查詢,牙科,明天,下午
  系統回應: 2026-10-30 下午 李醫師 目前剩餘 12 個名額。

測試編號 3: 【通過 ✅】
  使用者說: 我想取消張醫師今晚的門診
  預期: 動作=取消, 對象=張醫師
  AI 原始回覆: 取消,張醫師,今天,晚上
  系統解讀: 取消,張醫師,今天,晚上
  系統回應: 已成功為您取消 張醫師 的預約。

測試編號 4: 【通過 ✅】
  使用者說: 請問你們開到幾點？
  預期: 動作=未知, 對象=未知
  AI 原始回覆: 查詢,未知,未知,未知
  系統解讀: 查詢,未知,未知,未知
  系統回應: 無法辨識需求，請說明欲掛號的科別與日期時段

測試編號 5: 【通過 ✅】
  使用者說: 我肚子很痛該吃什麼藥？
  預期: 動作=未知, 對象=未知
  AI 原始回覆: 未知,未知,未知,未知
  系統解讀: 未知,未知,未知,未知
  系統回應: 無法提供醫療建議，若身體不適請儘速就醫

測試編號 6: 【通過 ✅】
  使用者說: 我要掛號明天早上的外科
  預期: 動作=掛號, 對象=外科
  AI 原始回覆: 掛號,外科,明天,早上
  系統解讀: 掛號,外科,明天,早上
  系統回應: 該診次名額已滿，請選擇其他時段或日期。

測試編號 7: 【通過 ✅】
  使用者說: 我要取消後天的內科預約
  預期: 動作=取消, 對象=內科
  AI 原始回覆: 取消,內科,後天,未知
  系統解讀: 取消,內科,後天,未知
  系統回應: 已成功為您取消 內科 的預約。

測試編號 8: 【通過 ✅】
  使用者說: 幫我掛號下禮拜八的骨科
  預期: 動作=未知, 對象=未知
  AI 原始回覆: 掛號,未知,未知,未知
  系統解讀: 掛號,未知

In [8]:
# ============================================================
# 第 8 格：Gradio 網頁
import gradio as gr

def gradio_chatbot(user_input):
    if not user_input.strip():
        return "請輸入您的掛號或查詢需求。"
    _, _, system_reply = handle_system_logic(user_input)
    return system_reply

# 建立 Gradio 介面
demo = gr.Interface(
    fn=gradio_chatbot,
    inputs=gr.Textbox(lines=2, placeholder="請輸入您的需求（例如：我要掛號今天的內科）..."),
    outputs=gr.Textbox(label="系統回應"),
    title="診所智慧掛號系統",
    description="歡迎使用診所掛號系統！您可以進行門診查詢、預約掛號與取消預約。"
)

# 啟動網頁
demo.launch(inline=False)
# 提醒：放在最後一格；GitHub 上看不到網頁，批改看的是第 5、7 格的結果
# ============================================================
# （把 AI 產生、你看過並理解的程式貼在這裡）

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://abb09510a70bd24872.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# ✅ 交件前檢查清單（每一項都要打勾）

- [ ] 程式裡**沒有**任何 `gsk_` 開頭的金鑰，也沒有 print 出金鑰
- [ ] 規格書第 1～7 節都填完，測試清單至少 8 句
- [ ] 選單「執行階段 → 重新啟動並執行全部」，**全部跑完沒有紅色錯誤**
- [ ] 每一格左邊的執行編號是**連續遞增**的（1、2、3…）
- [ ] 選單「編輯 → 筆記本設定」，**沒有**勾選「儲存筆記本時忽略程式碼儲存格輸出內容」
- [ ] 選單「檔案 → 在 GitHub 中儲存副本」，存到**公開（Public）**的 repo
- [ ] 用無痕視窗打開 GitHub 上的檔案，確認看得到程式和執行結果